# 오케스트레이터-워커(Orchestrator-Workers) 워크플로

## 들어가며

같은 작업에 여러 관점이 필요한데 어떤 관점이 가장 가치 있을지 미리 알 수 없었던 적이 있으신가요? 오케스트레이터-워커 패턴은 중앙 LLM이 각 작업을 분석해 전문 워커 LLM에 위임할 최적의 하위 작업을 동적으로 결정하는 방식으로 이를 해결합니다.

전통적인 방식은 사람이 여러 번 프롬프트를 주거나, 맥락과 무관하게 늘 같은 변형을 만들어 내는 하드코딩된 병렬화를 쓰게 됩니다.

이 방식에서는 오케스트레이터 LLM이 작업을 분석해 이번 사례에 어떤 변형이 가장 가치 있을지 정한 뒤, 각 변형을 생성할 워커 LLM에 위임합니다.

### 만들 것

제품 설명 요청을 받아 다음을 수행하는 시스템입니다.

1. 어떤 종류의 마케팅 문구가 가치 있을지 분석합니다
2. 워커를 위한 전문화된 작업 설명을 동적으로 생성합니다
3. 서로 다른 청중에 맞춘 여러 콘텐츠 변형을 만들어 냅니다
4. 모든 워커의 결과를 조율해 반환합니다

### 사전 준비

- Python 3.9 이상
- 환경 변수로 설정한 Anthropic API 키: `export ANTHROPIC_API_KEY='your-key'`
- 프롬프트 엔지니어링에 대한 기본 이해
- Python 클래스와 타입 힌트에 대한 친숙함


### 이 워크플로를 언제 쓸까

필요한 하위 작업을 미리 예측할 수 없는 복잡한 작업에 잘 맞습니다. 단순 병렬화와의 핵심 차이는 유연성입니다. 하위 작업이 미리 정의되지 않고, 구체적인 입력을 바탕으로 오케스트레이터가 결정합니다.

**이 패턴을 쓸 때:**

- 작업에 서로 다른 여러 접근이나 관점이 필요할 때
- 최적의 하위 작업이 구체적인 입력에 따라 달라질 때
- 서로 다른 전략이나 스타일을 비교해야 할 때

**이 패턴을 쓰지 말아야 할 때:**

- 출력이 하나뿐인 단순한 작업일 때(불필요한 복잡성)
- 지연 시간이 결정적일 때(여러 번의 LLM 호출이 부담을 더합니다)
- 하위 작업이 예측 가능해 미리 정의할 수 있을 때(더 단순한 병렬화를 쓰세요)

## 동작 방식

오케스트레이터-워커 패턴은 두 단계로 동작합니다.

1. **분석 및 계획 단계**: 오케스트레이터 LLM이 작업과 맥락을 받아 어떤 접근이 가치 있을지 분석하고, 구조화된 하위 작업 설명을 XML 형식으로 생성합니다.

2. **실행 단계**: 각 워커 LLM은 다음을 받습니다.
   - 맥락을 위한 원래 작업
   - 자신의 하위 작업 유형과 설명
   - 함께 제공된 추가 맥락

오케스트레이터가 *실행 시점에* 어떤 하위 작업을 만들지 정하므로, 미리 정의된 병렬 워크플로보다 적응적입니다.

## 준비

### 설치
```bash
pip install anthropic
```

### 헬퍼 함수
이 예제는 LLM 호출과 XML 응답 파싱을 위해 `util.py`의 헬퍼 함수를 사용합니다.

- `llm_call(prompt, system_prompt="", model="claude-sonnet-4-6")`: Claude에 프롬프트를 보내고 텍스트 응답을 반환합니다
- `extract_xml(text, tag)`: 정규식으로 XML 태그에서 내용을 추출합니다

이 유틸리티들이 API 인증(환경에서 `ANTHROPIC_API_KEY` 읽기)을 처리하고 오케스트레이터-워커 패턴을 위한 간단한 인터페이스를 제공합니다. 전체 구현은 [util.py](util.py)에서 볼 수 있습니다.

## 구현

`FlexibleOrchestrator` 클래스가 두 단계 워크플로를 조율합니다.

**핵심 설계 결정:**
- 프롬프트는 유연성을 위해 실행 시점 변수(`task`, `context`)를 받는 템플릿입니다
- 구조화된 출력 파싱에 XML을 사용합니다(신뢰할 수 있고 언어 모델에 친화적인 형식)
- 워커는 더 나은 맥락을 위해 원래 작업과 자신의 지시를 모두 받습니다
- 오류 처리로 워커가 빈 응답을 반환하지 않았는지 검증합니다

구현에는 다음이 포함됩니다.
- `parse_tasks()`: 오케스트레이터의 XML 출력을 구조화된 작업 딕셔너리로 파싱합니다
- `FlexibleOrchestrator.process()`: 오케스트레이터를 호출한 뒤 워커를 호출하는 주 조율 로직입니다
- 워커의 빈 출력을 잡아 처리하는 응답 검증

In [ ]:
from util import extract_xml, llm_call

# Model configuration
MODEL = "claude-sonnet-4-6"  # Fast, capable model for both orchestrator and workers


def parse_tasks(tasks_xml: str) -> list[dict]:
    """Parse XML tasks into a list of task dictionaries."""
    tasks = []
    current_task = {}

    for line in tasks_xml.split("\n"):
        line = line.strip()
        if not line:
            continue

        if line.startswith("<task>"):
            current_task = {}
        elif line.startswith("<type>"):
            current_task["type"] = line[6:-7].strip()
        elif line.startswith("<description>"):
            current_task["description"] = line[12:-13].strip()
        elif line.startswith("</task>"):
            if "description" in current_task:
                if "type" not in current_task:
                    current_task["type"] = "default"
                tasks.append(current_task)

    return tasks


class FlexibleOrchestrator:
    """Break down tasks and run them in parallel using worker LLMs."""

    def __init__(
        self,
        orchestrator_prompt: str,
        worker_prompt: str,
        model: str = MODEL,
    ):
        """Initialize with prompt templates and model selection."""
        self.orchestrator_prompt = orchestrator_prompt
        self.worker_prompt = worker_prompt
        self.model = model

    def _format_prompt(self, template: str, **kwargs) -> str:
        """Format a prompt template with variables."""
        try:
            return template.format(**kwargs)
        except KeyError as e:
            raise ValueError(f"Missing required prompt variable: {e}") from e

    def process(self, task: str, context: dict | None = None) -> dict:
        """Process task by breaking it down and running subtasks in parallel."""
        context = context or {}

        # Step 1: Get orchestrator response
        orchestrator_input = self._format_prompt(self.orchestrator_prompt, task=task, **context)
        orchestrator_response = llm_call(orchestrator_input, model=self.model)

        # Parse orchestrator response
        analysis = extract_xml(orchestrator_response, "analysis")
        tasks_xml = extract_xml(orchestrator_response, "tasks")
        tasks = parse_tasks(tasks_xml)

        print("\n" + "=" * 80)
        print("ORCHESTRATOR ANALYSIS")
        print("=" * 80)
        print(f"\n{analysis}\n")

        print("\n" + "=" * 80)
        print(f"IDENTIFIED {len(tasks)} APPROACHES")
        print("=" * 80)
        for i, task_info in enumerate(tasks, 1):
            print(f"\n{i}. {task_info['type'].upper()}")
            print(f"   {task_info['description']}")

        print("\n" + "=" * 80)
        print("GENERATING CONTENT")
        print("=" * 80 + "\n")

        # Step 2: Process each task
        worker_results = []
        for i, task_info in enumerate(tasks, 1):
            print(f"[{i}/{len(tasks)}] Processing: {task_info['type']}...")

            worker_input = self._format_prompt(
                self.worker_prompt,
                original_task=task,
                task_type=task_info["type"],
                task_description=task_info["description"],
                **context,
            )

            worker_response = llm_call(worker_input, model=self.model)
            worker_content = extract_xml(worker_response, "response")

            # Validate worker response - handle empty outputs
            if not worker_content or not worker_content.strip():
                print(f"⚠️  Warning: Worker '{task_info['type']}' returned no content")
                worker_content = f"[Error: Worker '{task_info['type']}' failed to generate content]"

            worker_results.append(
                {
                    "type": task_info["type"],
                    "description": task_info["description"],
                    "result": worker_content,
                }
            )

        # Display results
        print("\n" + "=" * 80)
        print("RESULTS")
        print("=" * 80)
        for i, result in enumerate(worker_results, 1):
            print(f"\n{'-' * 80}")
            print(f"Approach {i}: {result['type'].upper()}")
            print(f"{'-' * 80}")
            print(f"\n{result['result']}\n")

        return {
            "analysis": analysis,
            "worker_results": worker_results,
        }

## 사용 예시: 마케팅 문구 변형 생성

이제 오케스트레이터-워커 패턴이 실제로 동작하는 모습을 실용적인 예제로 살펴봅니다. 제품에 대한 여러 스타일의 마케팅 문구를 생성하는 것입니다.

**이 예제가 패턴을 잘 보여 주는 이유:**
- 제품마다 효과적인 마케팅 각도가 다릅니다
- "최선의" 변형은 구체적인 제품 특징과 목표 청중에 따라 달라집니다
- 오케스트레이터가 고정된 템플릿 대신 입력에 맞춰 전략을 조정할 수 있습니다

**프롬프트 설계 참고:**
- 오케스트레이터 프롬프트는 2~3가지 접근을 요청하고 XML 구조 안내를 제공합니다
- 워커 프롬프트는 워커에 전체 맥락(원래 작업, 자신의 스타일, 지침)을 제공합니다
- 두 프롬프트 모두 안정적인 파싱을 위해 명확한 XML 형식을 사용합니다

In [9]:
ORCHESTRATOR_PROMPT = """
Analyze this task and break it down into 2-3 distinct approaches:

Task: {task}

Return your response in this format:

<analysis>
Explain your understanding of the task and which variations would be valuable.
Focus on how each approach serves different aspects of the task.
</analysis>

<tasks>
    <task>
    <type>formal</type>
    <description>Write a precise, technical version that emphasizes specifications</description>
    </task>
    <task>
    <type>conversational</type>
    <description>Write an engaging, friendly version that connects with readers</description>
    </task>
</tasks>
"""

WORKER_PROMPT = """
Generate content based on:
Task: {original_task}
Style: {task_type}
Guidelines: {task_description}

Return your response in this format:

<response>
Your content here, maintaining the specified style and fully addressing requirements.
</response>
"""


orchestrator = FlexibleOrchestrator(
    orchestrator_prompt=ORCHESTRATOR_PROMPT,
    worker_prompt=WORKER_PROMPT,
)

results = orchestrator.process(
    task="Write a product description for a new eco-friendly water bottle",
    context={
        "target_audience": "environmentally conscious millennials",
        "key_features": ["plastic-free", "insulated", "lifetime warranty"],
    },
)


ORCHESTRATOR ANALYSIS


This task requires creating marketing copy for an eco-friendly water bottle. The core challenge is balancing product information with persuasive messaging while highlighting the environmental benefits. Different approaches would serve distinct marketing channels and audience segments:

1. A **feature-focused technical approach** would appeal to detail-oriented consumers who make purchasing decisions based on specifications, materials, and measurable environmental impact. This serves e-commerce listings and comparison shopping.

2. A **lifestyle-oriented emotional approach** would connect with values-driven consumers through storytelling and aspirational messaging, emphasizing how the product fits into an eco-conscious lifestyle. This serves social media and brand-building content.

3. A **benefit-driven practical approach** would focus on solving everyday problems while weaving in sustainability advantages, appealing to mainstream consumers who want both functi

## 정리

구체적인 입력에 따라 작업 분해를 동적으로 조정하는 오케스트레이터-워커 패턴을 구현해 봤습니다. 이 패턴은 어떤 변형이 필요한지 미리 정의하지 않고도, 각기 다른 청중과 맥락에 맞춘 여러 마케팅 문구 변형을 만들어 냈습니다.

### 핵심 정리

**패턴의 장점:**
- **적응성**: 오케스트레이터가 입력마다 최적의 접근을 결정합니다
- **유연성**: 프롬프트만 바꾸면 다른 영역에도 쉽게 적용됩니다
- **구조화된 조율**: XML 기반 통신이 안정적인 파싱을 보장합니다
- **오류 내성**: 검증이 워커 실패를 잡아 처리합니다

**이 패턴이 빛나는 때:**
- 여러 관점이 필요한 콘텐츠 생성(마케팅, 문서화, 창작 글쓰기)
- 서로 다른 분석 렌즈가 도움이 되는 분석 작업
- 분해 전략이 문제에 따라 달라지는 문제 해결

### 한계와 고려 사항

**비용과 지연 시간:**
- LLM 호출이 N+1번 필요합니다(오케스트레이터 1 + 워커 N)
- 이 구현은 순차 처리입니다(워커가 한 번에 하나씩 실행)
- 성능을 높이려면 `asyncio`나 스레드 풀로 워커 호출을 병렬화하는 것을 고려하세요

**이 패턴을 쓰지 말아야 할 때:**
- 출력이 하나로 분명한 단순 작업(추가된 복잡성이 정당화되지 않습니다)
- 지연 시간이 결정적인 애플리케이션(여러 API 호출이 부담을 더합니다)
- 하위 작업이 늘 동일한 작업(미리 정의된 병렬화를 쓰세요)

**고려할 실패 양상:**
- 오케스트레이터가 작업을 최적으로 분해하지 못할 수 있습니다(프롬프트 엔지니어링이 결정적입니다)
- 워커가 비어 있거나 잘못된 형식의 응답을 반환할 수 있습니다(검증으로 처리합니다)
- 모델이 형식을 정확히 따르지 않으면 XML 파싱이 실패할 수 있습니다(JSON을 대안으로 고려해 보세요)

### 다음 단계

**이 구현 개선하기:**
1. 성능을 위해 `asyncio`로 워커를 병렬 실행하세요
2. 실패한 워커에 대한 재시도 로직을 구현하세요
3. LLM이 워커 출력을 합치는 종합 단계를 추가하세요
4. 서로 다른 오케스트레이터 전략을 실험해 보세요(예: 하위 작업을 더 많이/적게 요청하기)

**여러분의 사용 사례에 맞추기:**
- 여러분의 영역에 맞게 작업 분해를 이끌도록 오케스트레이터 프롬프트를 수정하세요
- 영역에 특화된 지시를 주도록 워커 프롬프트를 조정하세요
- 애플리케이션에 맞는 맥락 파라미터를 추가하세요
- 비용 대비 품질을 최적화하려면 오케스트레이터에 Claude Opus를, 워커에 Claude Haiku를 쓰는 것을 고려하세요